In [ ]:
# Import packages, load the study configuration, the passing samples and the gene
# constraint table, and set the input and output locations
from pathlib import Path

from SSc_Somatic import configure, load_config, load_study
from SSc_Somatic.burden import vcf_processing as vcf_funcs
from SSc_Somatic.loading import load_blood_sample_ids

%matplotlib inline

# Study configuration. Point this at another configuration file to analyze a different cohort.
config = configure(load_config(Path('..') / 'config' / 'example_SSc.yaml'))
data = load_study(config, 'mutation')

# Passing samples with the configured exclusions applied, and the gene constraint table
design_df = data.design
constraint_df = data.gene_constraint

annovar_vcf_filt_root = config.path('mutation', 'annovar_filtered_dir')
comb_filename = config.output_path('mutation', 'combined_variants')
tidy_vcf_filename = config.output_path('mutation', 'tidy_variants')

In [ ]:
# Restrict to samples sequenced from blood
blood_ids = load_blood_sample_ids(config)
design_df = design_df[design_df['sample_id'].isin(blood_ids)]
sample_list = design_df['sample_id'].tolist()
print(f"Number of blood samples included in the variant table: {len(sample_list)}")

In [ ]:
# Process all ANNOVAR-annotated VCFs into a combined variants dictionary
dbsnp_cosmic_filt = False

print("Reading the annotated VCF of each sample:")
variants, samples = vcf_funcs.process_vcfs(
    str(annovar_vcf_filt_root), comb_filename, sample_list,
    dbsnp_cosmic_filt=dbsnp_cosmic_filt,
    file_prefix=config.require('mutation', 'annovar_file_prefix'),
    vcf_suffix=config.require('mutation', 'annovar_vcf_suffix'),
    filtered_vcf_suffix=config.require('mutation', 'annovar_filtered_vcf_suffix')
)

In [ ]:
# Convert to dataframe and standardize columns
variants_df = vcf_funcs.variants_to_dataframe(variants, samples)

# Make unique identifier labels
variants_df.columns = ['CHR', 'POS', 'sample_id', 'REF', 'ALT', 'GENE', 'FUNC', 'exac_freq',
                   'CADD_PHRED', 'SIFT', 'POLYPHEN', 'pLI', 'gnomad_exome_freq', 'vaf', 'ad', 'sb',
                   'aachange_gene', 'aachange_transcript', 'aachange_exon', 'aachange_cdna', 'aachange_protein',
                       'aachange_full', 'clnalleleid', 'clndn', 'clndisdb', 'clnrevstat', 'clnsig',
                       'intervar_automated', 'pvs1', 'ps1', 'ps2', 'ps3', 'ps4', 'pm1', 'pm2', 'pm3', 'pm4',
                       'pm5', 'pm6', 'pp1', 'pp2', 'pp3', 'pp4', 'pp5', 'ba1', 'bs1', 'bs2', 'bs3', 'bs4',
                       'bp1', 'bp2', 'bp3', 'bp4', 'bp5', 'bp6', 'bp7']

variants_df['variant_id'] = variants_df['CHR'].str.replace('chr', '') + '-' + variants_df['POS'].astype(str) + '-' + variants_df['REF'] + '-' + variants_df['ALT']

# Save to file
variants_df.to_csv(comb_filename)

# Show the size of the combined table and the number of variants by exonic function
print("Combined variant table (rows, columns):", variants_df.shape)
print("Number of variants by exonic function in the combined table:")
print(variants_df['FUNC'].value_counts())

In [ ]:
# Filter combined dataframe to tidy format, removing intronic variants, doublets, and those without 1 strand each direction.
# The duplicate filter keeps the first row for each variant (chromosome, position, ref, alt) across all
# samples, so a variant found in several samples is kept for only one of them.
print("Filtering the combined table into the tidy variant table:")
tidy_df = vcf_funcs.filter_comb_to_tidy(
    variants_df,
    save_filename=tidy_vcf_filename,
    exclude_synonymous=False,
    exclude_unknown=False,
    exclude_intronic=False,
    exclude_doublets=False,
    min_strand_support=1,
    verbose=True,
    duplicate_columns=['variant_id']
)

In [ ]:
# Compare the exonic function classes after filtering with those in the combined table
print("Number of variants by exonic function in the tidy table:")
print(tidy_df['FUNC'].value_counts())
print("Exonic function classes in the combined table before filtering:")
print(variants_df['FUNC'].unique())

In [ ]:
# Add constraint metrics from gnomad per variant
print("Adding gene pLI scores and saving the tidy variant table:")
tidy_df = vcf_funcs.add_pli_to_dataframe(tidy_df, constraint_df)
tidy_df.to_csv(tidy_vcf_filename, index=False)